# Telecom Churn Prediction Project

Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

## Initial data preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
df = pd.read_csv('telco-churn.csv')
df.head()

In [ ]:
df.head().T

In [ ]:
df.columns = df.columns.str.lower()
df.columns

In [ ]:
df.dtypes

In [ ]:
categorical_columns = df.select_dtypes(include='string').columns.to_list()
df[categorical_columns] = df[categorical_columns].apply(lambda x: x.str.lower().str.replace(' ', '_', regex=False))
df.head().T

In [ ]:
df.isnull().sum()

In [ ]:
df['totalcharges'] = df['totalcharges'].apply(pd.to_numeric, errors='coerce')
df['seniorcitizen'] = df['seniorcitizen'].astype('str')
df.dtypes

In [ ]:
df[df['totalcharges'].isnull()][['customerid', 'totalcharges']]

In [ ]:
df['totalcharges'] = df['totalcharges'].fillna(0)

In [ ]:
df.churn.head()

In [ ]:
df['churn'] = (df.churn == 'yes').astype('int')
df.churn.head()

In [ ]:
df.head().T

## Validation framework

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=1)

In [ ]:
len(df_train), len(df_val), len(df_test)

In [ ]:
y_train = df_train.churn
y_val = df_val.churn
y_test = df_test.churn

In [ ]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

## Exploratory data analysis

In [ ]:
churn_rate = round(df_train_full.churn.mean(),2)
churn_rate

In [ ]:
numerical_columns = df_train_full.columns[(df_train_full.dtypes=='float') | (df_train_full.dtypes=='int')].tolist()
numerical_columns.remove('churn')
numerical_columns

In [ ]:
categorical_columns = df_train_full.columns[df_train_full.dtypes=='str'].tolist()
categorical_columns.remove('customerid')
categorical_columns

In [ ]:
df_train_full[categorical_columns].head()

In [ ]:
df_train_full[categorical_columns].nunique()

## Feature importance

### Categorical

In [ ]:
global_churn = df_train_full.churn.mean()

In [ ]:
df_train_full.groupby('gender').churn.mean()

In [ ]:
df_train_full.groupby('gender').churn.agg(['mean', 'count'])

In [ ]:
df_train_full.groupby('partner').churn.mean()

In [ ]:
for col in categorical_columns:
      print(col)
      df_group = df_train_full.groupby(col).churn.agg(['mean'])
      df_group['risk'] = df_group['mean'] / global_churn
      print(df_group)
      print()

In [ ]:
from sklearn.metrics import mutual_info_score

df_train_full[categorical_columns].apply(lambda x: mutual_info_score(x, df_train_full.churn)).sort_values(ascending=False)

### Numerical

In [ ]:
df_train_full[numerical_columns].corrwith(df_train_full.churn).abs()

## One-hot encoding

In [ ]:
df_train_full.head()

In [ ]:
y_train_full = df_train_full.churn.values

In [ ]:
df_train_one_hot_encoded = pd.get_dummies(df_train_full[categorical_columns], dtype=int)
df_train_full = pd.concat([df_train_full[numerical_columns], df_train_one_hot_encoded], axis=1)
df_train_full.head()

In [ ]:
df_train_one_hot_encoded = pd.get_dummies(df_train[categorical_columns], dtype=int)
df_train = pd.concat([df_train[numerical_columns], df_train_one_hot_encoded], axis=1)
df_train.head()

In [ ]:
df_val_one_hot_encoded = pd.get_dummies(df_val[categorical_columns], dtype=int)
df_val = pd.concat([df_val[numerical_columns], df_val_one_hot_encoded], axis=1)
df_val.head()

In [ ]:
df_test_one_hot_encoded = pd.get_dummies(df_test[categorical_columns], dtype=int)
df_test = pd.concat([df_test[numerical_columns], df_test_one_hot_encoded], axis=1)
df_test.head()

## Training logistic regression

In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [ ]:
z = np.linspace(10,-10,100)

In [ ]:
plt.plot(z, sigmoid(z))

In [ ]:
def train_logistic_regression(X, y):
    XTX = np.dot(X.T, X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w

In [ ]:
def predict_logistic_regression(X, w):
      return sigmoid(np.dot(X,w))

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(df_train, y_train)

In [ ]:
y_probs = model.predict_proba(df_val[:10])[:,1]
y_probs

In [ ]:
y_pred = (y_probs > 0.5).astype(int)

In [ ]:
(y_pred == y_val[:10]).mean()

In [ ]:
model.coef_

## Using the model

In [ ]:
df_train_full.head()

In [ ]:
y_train_full

In [ ]:
model_full = LogisticRegression()
model_full.fit(df_train_full, y_train_full)

In [ ]:
model_full.coef_

In [ ]:
y_pred = model_full.predict(df_test)

In [ ]:
(y_test == y_pred).mean()

In [ ]:
customer10 = df_test.iloc[10]
model_full.predict(np.array(customer10).reshape(1,-1))

In [ ]:
y_test.iloc[10]